# Preprocessing Demo - placement.csv

**Steps**
1. Load and look
2. Separate numeric and categorical columns
3. Split into X and y
4. Handle missing values
5. Encode categorical columns
6. Scale numeric columns
7. Do 5 and 6 together with ColumnTransformer

**Target shape at the end: (550, 14)**

In [29]:
import numpy as np
import pandas as pd

---
## Step 1 - Load and look

In [30]:
df = pd.read_csv('placement.csv')

In [31]:
df.head(1)

,student_id,cgpa,twelfth_percent,branch,internships,projects,backlogs,prep_hours,communication_score,hostel,placed,package_lpa
0,VS2821,7.41,78.5,IT,0.0,6,1,188.0,4.3,No,No,0.0


In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   student_id           550 non-null    str    
 1   cgpa                 550 non-null    float64
 2   twelfth_percent      550 non-null    float64
 3   branch               550 non-null    str    
 4   internships          519 non-null    float64
 5   projects             550 non-null    int64  
 6   backlogs             550 non-null    int64  
 7   prep_hours           454 non-null    float64
 8   communication_score  538 non-null    float64
 9   hostel               550 non-null    str    
 10  placed               550 non-null    str    
 11  package_lpa          517 non-null    float64
dtypes: float64(6), int64(2), str(4)
memory usage: 51.7 KB


In [33]:
df.describe().columns # returns only numerical columns

Index(['cgpa', 'twelfth_percent', 'internships', 'projects', 'backlogs',
       'prep_hours', 'communication_score', 'package_lpa'],
      dtype='str')

---
## Step 2 - Separate numeric and categorical columns

In [34]:
nums_col = [
    'cgpa', 'twelfth_percent', 'internships', 'projects', 'backlogs',
       'prep_hours', 'communication_score'
]

In [35]:
cat_col = [
    'branch','hostel'
]

---
## Step 3 - Split into X and y

In [36]:
X = df.drop(columns=['student_id','placed','package_lpa'])

In [37]:
y = df['placed']

---
## Step 4 - Handle missing values

In [38]:
print("Before filling numerical columns' null values: ", 
      df.isnull().sum())

Before filling numerical columns' null values:  student_id              0
cgpa                    0
twelfth_percent         0
branch                  0
internships            31
projects                0
backlogs                0
prep_hours             96
communication_score    12
hostel                  0
placed                  0
package_lpa            33
dtype: int64


In [51]:
for col in nums_col:
    X[col]= X[col].fillna(df[col].mean())

In [52]:
print("After filling numerical columns' null values: ", 
      df.isnull().sum())

After filling numerical columns' null values:  student_id              0
cgpa                    0
twelfth_percent         0
branch                  0
internships             0
projects                0
backlogs                0
prep_hours              0
communication_score     0
hostel                  0
placed                  0
package_lpa            33
dtype: int64


---
## Step 5 - Encode categorical columns

In [53]:
df['branch'].value_counts()

branch
CSE      167
IT       118
ECE      112
Mech     103
Civil     50
Name: count, dtype: int64

In [54]:
df['hostel'].value_counts()

hostel
Yes    317
No     233
Name: count, dtype: int64

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(X[cat_col]) 
encoded.shape # 7 columns bcs 5 from 'branch' and 2 from 'hostel'
# 'cat__branch_CSE', 'cat__branch_Civil', 'cat__branch_ECE',
# 'cat__branch_IT', 'cat__branch_Mech', 'cat__hostel_No',
# 'cat__hostel_Yes'


(550, 7)

---
## Step 6 - Scale numeric columns

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_col = scaler.fit_transform(X[nums_col]) # Scaler doensn't create any new columns for the numerical columns, it only scales the values, so the 7 columns in the input of nums_col remains 7
scaled_col.shape # 'num__cgpa', 'num__twelfth_percent',
# 'num__internships', 'num__projects', 'num__backlogs',
# 'num__prep_hours', 'num__communication_score'

(550, 7)

---
## Step 7 - Both together with ColumnTransformer

In [57]:
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ('cat',OneHotEncoder(), cat_col),
    ('num',StandardScaler(), nums_col)
])

In [58]:
final_transform = preprocessor.fit_transform(X)

In [59]:
final_transform.shape

(550, 14)

In [61]:
preprocessor.get_feature_names_out()

array(['cat__branch_CSE', 'cat__branch_Civil', 'cat__branch_ECE',
       'cat__branch_IT', 'cat__branch_Mech', 'cat__hostel_No',
       'cat__hostel_Yes', 'num__cgpa', 'num__twelfth_percent',
       'num__internships', 'num__projects', 'num__backlogs',
       'num__prep_hours', 'num__communication_score'], dtype=object)